# STRAT-002 Last Year Performance

How did the strategy perform in the most recent period?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import json

# Load previous backtest results
with open('../data/strat002_backtest_results.json', 'r') as f:
    results = json.load(f)

trades = pd.DataFrame(results['trades'])
trades['entry_date'] = pd.to_datetime(trades['entry_date'])
trades['exit_date'] = pd.to_datetime(trades['exit_date'])

print("ALL TRADES")
print("="*100)
print(trades[['entry_date', 'exit_date', 'entry_price', 'exit_price', 'net_return', 'exit_reason']].to_string())

In [ ]:
# Filter for last year (2025)
one_year_ago = datetime(2025, 1, 1)

# Trades that were active in the last year
last_year_trades = trades[
    (trades['exit_date'] >= one_year_ago) | 
    (trades['entry_date'] >= one_year_ago)
]

print("\nTRADES ACTIVE IN 2025")
print("="*100)
print(f"{'Entry':<12} {'Exit':<12} {'Entry $':>10} {'Exit $':>10} {'Return':>10} {'Exit Reason':<15}")
print("-"*100)

for _, t in last_year_trades.iterrows():
    print(f"{str(t['entry_date'].date()):<12} {str(t['exit_date'].date()):<12} "
          f"{t['entry_price']:>10,.0f} {t['exit_price']:>10,.0f} "
          f"{t['net_return']*100:>+9.1f}% {t['exit_reason']:<15}")

In [ ]:
# Calculate last year performance
if len(last_year_trades) > 0:
    total_return = (1 + last_year_trades['net_return']).prod() - 1
    win_rate = (last_year_trades['net_return'] > 0).mean()
    
    print("\n" + "="*60)
    print("LAST YEAR (2025) PERFORMANCE")
    print("="*60)
    print(f"\nTrades: {len(last_year_trades)}")
    print(f"Win Rate: {win_rate*100:.0f}%")
    print(f"Total Return: {total_return*100:+.1f}%")
    print(f"Avg Return: {last_year_trades['net_return'].mean()*100:+.1f}%")
else:
    print("No trades in last year")

In [ ]:
# Compare to BTC buy & hold for same period
DATA_DIR = Path("../data/daily")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")

# Get prices for comparison period
if len(last_year_trades) > 0:
    start_date = last_year_trades['entry_date'].min()
    end_date = last_year_trades['exit_date'].max()
    
    # For 2025 specifically
    jan_2025 = price[price.index >= '2025-01-01'].iloc[0]['price']
    latest = price.iloc[-1]['price']
    
    bh_2025_return = (latest / jan_2025) - 1
    
    print(f"\nBUY & HOLD 2025 (Jan 1 - Now):")
    print(f"  BTC Jan 1: ${jan_2025:,.0f}")
    print(f"  BTC Now: ${latest:,.0f}")
    print(f"  Return: {bh_2025_return*100:+.1f}%")
    
    print(f"\n📊 COMPARISON:")
    print(f"  Strategy 2025: {total_return*100:+.1f}%")
    print(f"  Buy & Hold 2025: {bh_2025_return*100:+.1f}%")
    print(f"  Difference: {(total_return - bh_2025_return)*100:+.1f}%")

In [ ]:
# Current status - are we in a trade?
print("\n" + "="*60)
print("CURRENT STATUS")
print("="*60)

latest_trade = trades.iloc[-1]

if latest_trade['exit_reason'] == 'end_of_data':
    print(f"\n🔵 CURRENTLY IN TRADE")
    print(f"   Entry: {latest_trade['entry_date'].date()} at ${latest_trade['entry_price']:,.0f}")
    print(f"   Current Price: ${latest:,.0f}")
    current_pnl = (latest / latest_trade['entry_price']) - 1
    print(f"   Unrealized P&L: {current_pnl*100:+.1f}%")
    
    # Check MVRV
    mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
    current_mvrv = mvrv.iloc[-1]['mvrv']
    print(f"   Current MVRV: {current_mvrv:.2f}")
    
    if current_mvrv >= 2.0:
        print(f"   ⚠️ MVRV > 2.0 - Trailing stop ACTIVE")
    else:
        print(f"   Waiting for MVRV > 2.0 to activate trail")
else:
    print(f"\n⚪ NOT IN TRADE")
    print(f"   Last trade exited: {latest_trade['exit_date'].date()}")
    print(f"   Exit reason: {latest_trade['exit_reason']}")
    print(f"   Waiting for next entry signal...")

In [ ]:
# Check if entry signal is currently active
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

# Calculate RL z-score
rl_ma30 = realized_loss['realized_loss'].rolling(30).mean()
rl_std30 = realized_loss['realized_loss'].rolling(30).std()
rl_zscore = (realized_loss['realized_loss'] - rl_ma30) / rl_std30

latest_sopr = sopr.iloc[-1]['sopr']
latest_sth_sopr = sopr_sth.iloc[-1]['sopr_sth']
latest_rl_z = rl_zscore.iloc[-1]

print(f"\n" + "="*60)
print("CURRENT SIGNAL STATUS")
print("="*60)
print(f"\n  SOPR: {latest_sopr:.4f} {'✅ < 1' if latest_sopr < 1 else '❌ >= 1'}")
print(f"  STH-SOPR: {latest_sth_sopr:.4f} {'✅ < 1' if latest_sth_sopr < 1 else '❌ >= 1'}")
print(f"  RL Z-Score: {latest_rl_z:.2f} {'✅ > 0.5' if latest_rl_z > 0.5 else '❌ <= 0.5'}")

if latest_sopr < 1 and latest_sth_sopr < 1 and latest_rl_z > 0.5:
    print(f"\n  🚨 ENTRY SIGNAL ACTIVE!")
else:
    print(f"\n  No entry signal currently")